# Insurance Risk Analytics: Exploratory Data Analysis

## Comprehensive Analysis of Insurance Portfolio Data

This notebook performs a detailed exploratory data analysis (EDA) of insurance claim data spanning an 18-month period. The analysis covers data quality assessment, statistical summaries, univariate and bivariate analysis, geographic trends, temporal patterns, and vehicle risk profiles.

### Objectives:
1. **Data Quality**: Assess data completeness and validity
2. **Distribution Analysis**: Understand patterns in premiums, claims, and customer values
3. **Segment Analysis**: Calculate loss ratios by province, vehicle type, and demographics
4. **Risk Profiling**: Identify high-risk vehicle makes, geographic regions, and customer segments
5. **Temporal Trends**: Analyze claim patterns over the 18-month period
6. **Actionable Insights**: Provide recommendations based on findings

**Analysis Date**: May 2026  
**Dataset**: MachineLearningRating_v3.txt

## 1. Import Libraries and Load Data

In [ ]:
# Import necessary libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import warnings
from datetime import datetime
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Configure visualization style
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11

# Suppress warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Load the insurance dataset
data_path = '../data/MachineLearningRating_v3.txt'

print("Loading insurance data...")
try:
    # Load with tab separator
    df = pd.read_csv(data_path, sep='\t')
    print(f"✓ Data loaded successfully!")
    print(f"  - Shape: {df.shape}")
    print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
except Exception as e:
    print(f"Error loading data: {e}")
    df = None

# Display basic information
if df is not None:
    print("\nDataset Overview:")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nColumn Names and Types:")
    print(df.dtypes)
    print(f"\nFirst few rows:")
    df.head()

## 2. Data Summarization and Quality Assessment

### 2.1 Data Quality Check

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print("Missing Value Analysis:")
print("=" * 70)
if len(missing_data) == 0:
    print("✓ No missing values detected!")
else:
    print(missing_data.to_string(index=False))
    print(f"\nHandling Strategy:")
    print("  - For numerical columns: Will use median imputation if needed")
    print("  - For categorical columns: Will use mode or create 'Unknown' category")

# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicate_count}")

# Data type summary
print(f"\nData Type Summary:")
print(df.dtypes.value_counts())

### 2.2 Descriptive Statistics for Numerical Columns

In [ ]:
# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical Columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical Columns ({len(categorical_cols)}): {categorical_cols}")

# Descriptive statistics
print("\n" + "="*100)
print("DESCRIPTIVE STATISTICS FOR NUMERICAL FEATURES")
print("="*100)

desc_stats = df[numerical_cols].describe().T
desc_stats['skewness'] = df[numerical_cols].skew()
desc_stats['kurtosis'] = df[numerical_cols].kurtosis()
desc_stats['cv'] = (df[numerical_cols].std() / df[numerical_cols].mean()).round(3)  # Coefficient of variation

print(desc_stats.round(3).to_string())

# Categorical columns summary
print("\n" + "="*100)
print("CATEGORICAL COLUMNS SUMMARY")
print("="*100)

for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Top 5 values:")
    print(df[col].value_counts().head().to_string())

## 3. Univariate Analysis

### 3.1 Distribution of Numerical Features

In [ ]:
# Select key numerical columns for visualization
key_numerical = [col for col in numerical_cols if col in ['TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'SumInsured']]

# Create histograms for key numerical columns
if len(key_numerical) > 0:
    fig, axes = plt.subplots(len(key_numerical), 2, figsize=(16, 5*len(key_numerical)))
    
    if len(key_numerical) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, col in enumerate(key_numerical):
        # Histogram
        axes[idx, 0].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
        axes[idx, 0].set_title(f'{col} - Histogram', fontsize=12, fontweight='bold')
        axes[idx, 0].set_xlabel(col)
        axes[idx, 0].set_ylabel('Frequency')
        axes[idx, 0].grid(True, alpha=0.3)
        
        # KDE plot
        df[col].dropna().plot(kind='density', ax=axes[idx, 1], color='steelblue', linewidth=2)
        axes[idx, 1].set_title(f'{col} - Density Plot', fontsize=12, fontweight='bold')
        axes[idx, 1].set_xlabel(col)
        axes[idx, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Numerical distributions visualized")
else:
    print("No key numerical columns found for visualization")

### 3.2 Distribution of Categorical Features

In [ ]:
# Select key categorical columns
key_categorical = [col for col in categorical_cols if df[col].nunique() <= 20][:6]

if len(key_categorical) > 0:
    fig, axes = plt.subplots((len(key_categorical) + 1) // 2, 2, figsize=(16, 4*(len(key_categorical)//2 + 1)))
    axes = axes.flatten()
    
    for idx, col in enumerate(key_categorical):
        value_counts = df[col].value_counts()
        axes[idx].bar(range(len(value_counts)), value_counts.values, color='steelblue', edgecolor='black')
        axes[idx].set_xticks(range(len(value_counts)))
        axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
        axes[idx].set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Count')
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Hide empty subplots
    for idx in range(len(key_categorical), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Categorical distributions visualized ({len(key_categorical)} columns)")
else:
    print("No categorical columns found for visualization")

## 4. Bivariate and Multivariate Analysis

### 4.1 Correlation Matrix

In [ ]:
# Calculate correlation matrix
if len(numerical_cols) > 0:
    corr_matrix = df[numerical_cols].corr()
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title('Correlation Matrix - Numerical Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Find strong correlations
    print("\nStrong Correlations (|r| > 0.5):")
    print("="*60)
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.5:
                print(f"{corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

### 4.2 Relationship between Premium and Claims

In [ ]:
# Check if key columns exist
if 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Scatter plot
    axes[0].scatter(df['TotalPremium'], df['TotalClaims'], alpha=0.5, s=30, color='steelblue')
    axes[0].set_xlabel('Total Premium')
    axes[0].set_ylabel('Total Claims')
    axes[0].set_title('Premium vs Claims Relationship', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Log scale version for better visibility
    df_nonzero = df[(df['TotalPremium'] > 0) & (df['TotalClaims'] > 0)]
    axes[1].scatter(df_nonzero['TotalPremium'], df_nonzero['TotalClaims'], 
                   alpha=0.5, s=30, color='steelblue')
    axes[1].set_xlabel('Total Premium (log scale)')
    axes[1].set_ylabel('Total Claims (log scale)')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('Premium vs Claims (Log Scale)', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Scatter plots created")
else:
    print("TotalPremium or TotalClaims columns not found")

## 5. Loss Ratio Analysis

### 5.1 Overall Loss Ratio

In [ ]:
# Calculate loss ratio (TotalClaims / TotalPremium)
if 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    total_premium = df['TotalPremium'].sum()
    total_claims = df['TotalClaims'].sum()
    overall_loss_ratio = total_claims / total_premium
    
    print("PORTFOLIO LOSS RATIO ANALYSIS")
    print("=" * 70)
    print(f"Total Premium:           ${total_premium:,.2f}")
    print(f"Total Claims:            ${total_claims:,.2f}")
    print(f"Overall Loss Ratio:      {overall_loss_ratio:.4f} ({overall_loss_ratio*100:.2f}%)")
    print(f"\nInterpretation:")
    if overall_loss_ratio < 0.5:
        print(f"  → EXCELLENT: Strong profitability (Loss Ratio < 50%)")
    elif overall_loss_ratio < 0.75:
        print(f"  → GOOD: Reasonable profitability (Loss Ratio 50-75%)")
    elif overall_loss_ratio < 1.0:
        print(f"  → FAIR: Marginal profitability (Loss Ratio 75-100%)")
    else:
        print(f"  → POOR: Unprofitable portfolio (Loss Ratio > 100%)")
else:
    print("Premium or Claims columns not found")

### 5.2 Loss Ratio by Segment (Province, VehicleType, Gender)

In [ ]:
# Segmented loss ratio analysis
segments = ['Province', 'VehicleType', 'Gender']
segment_analysis_results = {}

for segment in segments:
    if segment in df.columns:
        segment_data = df.groupby(segment, observed=True).agg({
            'TotalPremium': 'sum',
            'TotalClaims': 'sum'
        }).reset_index()
        
        segment_data['LossRatio'] = (segment_data['TotalClaims'] / segment_data['TotalPremium']).round(4)
        segment_data['RecordCount'] = df.groupby(segment, observed=True).size().values
        segment_data = segment_data.sort_values('LossRatio', ascending=False)
        
        segment_analysis_results[segment] = segment_data
        
        print(f"\nLOSS RATIO BY {segment.upper()}")
        print("=" * 80)
        print(segment_data.to_string(index=False))

# Create visualization for loss ratio by segment
if len(segment_analysis_results) > 0:
    fig, axes = plt.subplots(1, len(segment_analysis_results), figsize=(16, 5))
    
    if len(segment_analysis_results) == 1:
        axes = [axes]
    
    for idx, (segment, data) in enumerate(segment_analysis_results.items()):
        axes[idx].barh(data[segment], data['LossRatio'], color='steelblue', edgecolor='black')
        axes[idx].set_xlabel('Loss Ratio')
        axes[idx].set_title(f'Loss Ratio by {segment}', fontsize=12, fontweight='bold')
        axes[idx].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 6. Outlier Detection

### 6.1 Box Plots for Numerical Features

In [ ]:
# Box plots for outlier detection
if len(key_numerical) > 0:
    fig, axes = plt.subplots(1, len(key_numerical), figsize=(16, 6))
    
    if len(key_numerical) == 1:
        axes = [axes]
    
    for idx, col in enumerate(key_numerical):
        df.boxplot(column=col, ax=axes[idx])
        axes[idx].set_title(f'{col} - Box Plot', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Box plots created")

# Detect outliers using IQR method
print("\nOUTLIER DETECTION (IQR Method)")
print("=" * 70)

for col in key_numerical:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
    outlier_count = outliers.sum()
    outlier_pct = (outlier_count / len(df) * 100)
    
    print(f"\n{col}:")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers: {outlier_count} ({outlier_pct:.2f}%)")

## 7. Temporal Trends Analysis

### 7.1 Claims Frequency and Severity Over Time

In [ ]:
# Identify date columns
date_cols = []
for col in df.columns:
    if df[col].dtype == 'object':
        # Try to parse as date
        try:
            pd.to_datetime(df[col], errors='coerce')
            if df[col].notna().sum() > len(df) * 0.8:
                date_cols.append(col)
        except:
            pass

print(f"Date columns found: {date_cols}")

if len(date_cols) > 0:
    # Use the first date column for temporal analysis
    date_col = date_cols[0]
    df_temporal = df.copy()
    df_temporal[date_col] = pd.to_datetime(df_temporal[date_col], errors='coerce')
    df_temporal = df_temporal.dropna(subset=[date_col])
    
    # Extract temporal features
    df_temporal['Year'] = df_temporal[date_col].dt.year
    df_temporal['Month'] = df_temporal[date_col].dt.month
    df_temporal['YearMonth'] = df_temporal[date_col].dt.to_period('M')
    
    # Analyze trends
    if 'TotalClaims' in df_temporal.columns and 'TotalPremium' in df_temporal.columns:
        monthly_trends = df_temporal.groupby('YearMonth', observed=True).agg({
            'TotalPremium': ['sum', 'count'],
            'TotalClaims': ['sum', 'mean']
        }).reset_index()
        
        monthly_trends.columns = ['YearMonth', 'Premium_Sum', 'ClaimCount', 'Claims_Sum', 'Claims_Mean']
        monthly_trends['LossRatio'] = (monthly_trends['Claims_Sum'] / monthly_trends['Premium_Sum']).round(4)
        
        print(f"\nMONTHLY TRENDS (Sample):")
        print(monthly_trends.head(10).to_string(index=False))
        
        # Plot trends
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        
        # Premium trend
        axes[0, 0].plot(range(len(monthly_trends)), monthly_trends['Premium_Sum'], 
                       marker='o', color='steelblue', linewidth=2)
        axes[0, 0].set_title('Monthly Premium Trend', fontsize=12, fontweight='bold')
        axes[0, 0].set_ylabel('Total Premium')
        axes[0, 0].grid(True, alpha=0.3)
        
        # Claims trend
        axes[0, 1].plot(range(len(monthly_trends)), monthly_trends['Claims_Sum'], 
                       marker='o', color='darkred', linewidth=2)
        axes[0, 1].set_title('Monthly Claims Trend', fontsize=12, fontweight='bold')
        axes[0, 1].set_ylabel('Total Claims')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Claim frequency
        axes[1, 0].plot(range(len(monthly_trends)), monthly_trends['ClaimCount'], 
                       marker='o', color='orange', linewidth=2)
        axes[1, 0].set_title('Monthly Claim Frequency', fontsize=12, fontweight='bold')
        axes[1, 0].set_ylabel('Number of Claims')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Loss ratio trend
        axes[1, 1].plot(range(len(monthly_trends)), monthly_trends['LossRatio'], 
                       marker='o', color='green', linewidth=2)
        axes[1, 1].set_title('Monthly Loss Ratio Trend', fontsize=12, fontweight='bold')
        axes[1, 1].set_ylabel('Loss Ratio')
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No date columns found for temporal analysis")

## 8. Vehicle Make/Model Analysis

In [ ]:
# Vehicle analysis
vehicle_cols = [col for col in df.columns if 'Make' in col or 'Model' in col or 'VehicleType' in col or 'Auto' in col]

print(f"Vehicle-related columns: {vehicle_cols}")

if 'TotalClaims' in df.columns and len(vehicle_cols) > 0:
    print("\nVEHICLE ANALYSIS - CLAIM AMOUNTS")
    print("=" * 70)
    
    for col in vehicle_cols:
        if df[col].dtype == 'object' and df[col].nunique() < 50:
            vehicle_stats = df.groupby(col, observed=True).agg({
                'TotalClaims': ['count', 'sum', 'mean', 'median']
            }).round(2)
            
            vehicle_stats.columns = ['Count', 'Total_Claims', 'Mean_Claims', 'Median_Claims']
            vehicle_stats = vehicle_stats.sort_values('Mean_Claims', ascending=False)
            
            print(f"\n{col}:")
            print(vehicle_stats.to_string())
            
            # Visualize top and bottom
            top_bottom = pd.concat([
                vehicle_stats.head(5),
                vehicle_stats.tail(5)
            ])
            
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.barh(range(len(top_bottom)), top_bottom['Mean_Claims'], color='steelblue', edgecolor='black')
            ax.set_yticks(range(len(top_bottom)))
            ax.set_yticklabels(top_bottom.index)
            ax.set_xlabel('Mean Claim Amount')
            ax.set_title(f'Top and Bottom {col} by Mean Claim Amount', fontsize=12, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')
            plt.tight_layout()
            plt.show()

## 9. Geographic Trends Analysis

In [ ]:
# Geographic analysis
if 'Province' in df.columns:
    print("\nGEOGRAPHIC TRENDS - PROVINCE ANALYSIS")
    print("=" * 70)
    
    # Premium and claims by province
    province_stats = df.groupby('Province', observed=True).agg({
        'TotalPremium': ['sum', 'mean', 'count'],
        'TotalClaims': ['sum', 'mean']
    }).round(2)
    
    province_stats.columns = ['Premium_Total', 'Premium_Avg', 'RecordCount', 'Claims_Total', 'Claims_Avg']
    province_stats['LossRatio'] = (province_stats['Claims_Total'] / province_stats['Premium_Total']).round(4)
    province_stats = province_stats.sort_values('Premium_Total', ascending=False)
    
    print("\nProvince Statistics:")
    print(province_stats.to_string())
    
    # Visualizations by province
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Average premium by province
    province_premium = df.groupby('Province', observed=True)['TotalPremium'].mean().sort_values(ascending=False)
    axes[0, 0].barh(range(len(province_premium)), province_premium.values, color='steelblue', edgecolor='black')
    axes[0, 0].set_yticks(range(len(province_premium)))
    axes[0, 0].set_yticklabels(province_premium.index)
    axes[0, 0].set_xlabel('Average Premium')
    axes[0, 0].set_title('Average Premium by Province', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='x')
    
    # Average claims by province
    province_claims = df.groupby('Province', observed=True)['TotalClaims'].mean().sort_values(ascending=False)
    axes[0, 1].barh(range(len(province_claims)), province_claims.values, color='darkred', edgecolor='black')
    axes[0, 1].set_yticks(range(len(province_claims)))
    axes[0, 1].set_yticklabels(province_claims.index)
    axes[0, 1].set_xlabel('Average Claims')
    axes[0, 1].set_title('Average Claims by Province', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='x')
    
    # Record count by province
    province_count = df['Province'].value_counts()
    axes[1, 0].barh(range(len(province_count)), province_count.values, color='orange', edgecolor='black')
    axes[1, 0].set_yticks(range(len(province_count)))
    axes[1, 0].set_yticklabels(province_count.index)
    axes[1, 0].set_xlabel('Record Count')
    axes[1, 0].set_title('Records by Province', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='x')
    
    # Loss ratio by province
    loss_ratio_by_province = province_stats['LossRatio'].sort_values(ascending=False)
    axes[1, 1].barh(range(len(loss_ratio_by_province)), loss_ratio_by_province.values, color='green', edgecolor='black')
    axes[1, 1].set_yticks(range(len(loss_ratio_by_province)))
    axes[1, 1].set_yticklabels(loss_ratio_by_province.index)
    axes[1, 1].set_xlabel('Loss Ratio')
    axes[1, 1].set_title('Loss Ratio by Province', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
else:
    print("Province column not found")

## 10. Creative Visualizations and Key Insights

### 10.1 Insight Visualization #1: Multi-Dimensional Loss Ratio Dashboard

In [ ]:
# Creative Visualization #1: Multi-faceted Loss Ratio Heatmap
if 'VehicleType' in df.columns and 'Province' in df.columns and 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    # Create pivot table for heatmap
    pivot_data = df.groupby(['Province', 'VehicleType'], observed=True).agg({
        'TotalPremium': 'sum',
        'TotalClaims': 'sum'
    })
    
    pivot_data['LossRatio'] = (pivot_data['TotalClaims'] / pivot_data['TotalPremium']).round(3)
    
    # Reshape to heatmap format
    heatmap_data = pivot_data['LossRatio'].unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(heatmap_data.values, cmap='RdYlGn_r', aspect='auto')
    
    # Set ticks and labels
    ax.set_xticks(np.arange(len(heatmap_data.columns)))
    ax.set_yticks(np.arange(len(heatmap_data.index)))
    ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right')
    ax.set_yticklabels(heatmap_data.index)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Loss Ratio', rotation=270, labelpad=20)
    
    # Add text annotations
    for i in range(len(heatmap_data.index)):
        for j in range(len(heatmap_data.columns)):
            if heatmap_data.values[i, j] > 0:
                text = ax.text(j, i, f'{heatmap_data.values[i, j]:.2f}',
                             ha="center", va="center", color="black", fontsize=9, fontweight='bold')
    
    ax.set_title('Loss Ratio Heatmap: Province vs Vehicle Type', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Vehicle Type', fontsize=12)
    ax.set_ylabel('Province', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print("✓ Insight Visualization #1 created: Multi-Dimensional Loss Ratio Heatmap")

### 10.2 Insight Visualization #2: Premium vs Claims Bubble Chart by Province

In [ ]:
# Creative Visualization #2: Bubble Chart - Premium vs Claims by Province
if 'Province' in df.columns and 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    # Aggregate by Province
    bubble_data = df.groupby('Province', observed=True).agg({
        'TotalPremium': 'mean',
        'TotalClaims': 'mean',
        'Province': 'count'
    }).reset_index()
    bubble_data.columns = ['Province', 'AvgPremium', 'AvgClaims', 'RecordCount']
    
    # Create bubble chart
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create scatter plot with bubble sizes
    scatter = ax.scatter(bubble_data['AvgPremium'], 
                        bubble_data['AvgClaims'],
                        s=bubble_data['RecordCount']*2,  # Bubble size represents record count
                        alpha=0.6,
                        c=range(len(bubble_data)),
                        cmap='viridis',
                        edgecolors='black',
                        linewidth=2)
    
    # Add labels for each province
    for idx, row in bubble_data.iterrows():
        ax.annotate(row['Province'], 
                   (row['AvgPremium'], row['AvgClaims']),
                   fontsize=10, fontweight='bold',
                   ha='center', va='center')
    
    ax.set_xlabel('Average Premium ($)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Claims ($)', fontsize=12, fontweight='bold')
    ax.set_title('Premium vs Claims Analysis by Province\n(Bubble size = Number of Records)', 
                fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Insight Visualization #2 created: Premium vs Claims Bubble Chart")

### 10.3 Insight Visualization #3: Risk Matrix - Vehicle Type vs Gender

In [ ]:
# Creative Visualization #3: Risk Matrix with bubble sizes
if 'VehicleType' in df.columns and 'Gender' in df.columns and 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    # Create risk matrix data
    risk_data = df.groupby(['VehicleType', 'Gender'], observed=True).agg({
        'TotalPremium': ['sum', 'count'],
        'TotalClaims': 'sum'
    }).reset_index()
    
    risk_data.columns = ['VehicleType', 'Gender', 'Premium', 'Count', 'Claims']
    risk_data['LossRatio'] = (risk_data['Claims'] / risk_data['Premium']).round(3)
    
    # Create subplots for different views
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # View 1: Scatter by Vehicle Type with Gender coloring
    gender_colors = {'M': 'steelblue', 'F': 'coral', 'Male': 'steelblue', 'Female': 'coral'}
    
    for gender in risk_data['Gender'].unique():
        gender_data = risk_data[risk_data['Gender'] == gender]
        color = gender_colors.get(gender, 'gray')
        axes[0].scatter(gender_data['Premium'], 
                       gender_data['LossRatio'],
                       s=gender_data['Count']*5,
                       alpha=0.6,
                       label=f'Gender: {gender}',
                       color=color,
                       edgecolors='black',
                       linewidth=2)
    
    # Add labels
    for idx, row in risk_data.iterrows():
        axes[0].annotate(f"{row['VehicleType']}", 
                        (row['Premium'], row['LossRatio']),
                        fontsize=8, ha='center', va='center')
    
    axes[0].set_xlabel('Total Premium ($)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Loss Ratio', fontsize=11, fontweight='bold')
    axes[0].set_title('Risk Assessment: Premium vs Loss Ratio\n(Color=Gender, Bubble Size=Record Count)', 
                     fontsize=12, fontweight='bold')
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)
    
    # View 2: Stacked bar chart for risk comparison
    vehicle_gender = risk_data.groupby('VehicleType', observed=True).agg({
        'LossRatio': 'mean',
        'Count': 'sum'
    }).sort_values('LossRatio', ascending=False)
    
    axes[1].bar(range(len(vehicle_gender)), vehicle_gender['LossRatio'], 
               color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].set_xticks(range(len(vehicle_gender)))
    axes[1].set_xticklabels(vehicle_gender.index, rotation=45, ha='right')
    axes[1].set_ylabel('Average Loss Ratio', fontsize=11, fontweight='bold')
    axes[1].set_title('Average Loss Ratio by Vehicle Type', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(vehicle_gender['LossRatio']):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Insight Visualization #3 created: Risk Matrix - Vehicle Type vs Gender")

## 11. Key Findings and Actionable Insights

### Summary of Answers to Guiding Questions

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                    KEY FINDINGS AND RECOMMENDATIONS                           ║
╚════════════════════════════════════════════════════════════════════════════════╝

1. OVERALL LOSS RATIO & SEGMENTATION
   ────────────────────────────────────
   • The portfolio loss ratio provides a critical view of profitability
   • Segmented analysis reveals that certain provinces/vehicle types/demographics
     consistently outperform others
   • High loss ratio segments (>0.75) require immediate pricing review

2. FINANCIAL DISTRIBUTION & OUTLIERS
   ──────────────────────────────────
   • Key financial variables (TotalClaims, CustomValueEstimate) show skewed distributions
   • Outliers detected using IQR method should be investigated separately
   • Large outliers may indicate:
     - High-value claims that need special handling
     - Data entry errors requiring cleansing
     - Unique risk profiles deserving different pricing

3. TEMPORAL TRENDS
   ────────────────
   • Claim frequency and severity patterns help identify:
     - Seasonal risk variations
     - Long-term portfolio drift
     - Effectiveness of pricing changes
   • Monthly monitoring enables proactive portfolio management

4. VEHICLE RISK PROFILES
   ──────────────────────
   • Top claim vehicles: Focus on tighter underwriting, higher deductibles
   • Low claim vehicles: Consider competitive pricing to gain market share
   • Vehicle type analysis guides model-specific rating strategies

5. RECOMMENDATIONS
   ────────────────
   ✓ Implement dynamic pricing for high-loss segments
   ✓ Enhance underwriting criteria for risky vehicle types
   ✓ Conduct further analysis on temporal trends and seasonality
   ✓ Develop geographic pricing strategy based on loss ratios
   ✓ Create targeted retention programs for profitable segments

""")

### Conclusion

This comprehensive EDA has provided critical insights into the insurance portfolio's risk profile, profitability by segment, and geographic variations. The analysis forms a strong foundation for:

- **Pricing Strategy**: Segment-specific rate adjustments based on loss ratio analysis
- **Risk Management**: Identification and monitoring of high-risk profiles
- **Portfolio Optimization**: Focus on profitable segments and improvement of underperforming areas
- **Operational Efficiency**: Data quality improvements and process enhancements

The three creative visualizations effectively communicate complex, multi-dimensional relationships in the data, enabling stakeholders to quickly grasp key patterns and make informed decisions.

---
**Analysis Complete** | Generated: May 23, 2026